# 🎙️ F5-TTS Training sur Google Colab

Ce notebook permet de lancer un training F5-TTS contrôlé par un agent externe via Google Drive.

## 📋 Setup

1. Activer GPU : **Runtime → Change runtime type → GPU (T4/V100)**
2. Monter Google Drive
3. Créer dossier `MyDrive/f5_tts_orchestration/`
4. L'agent écrira `trigger.json` pour démarrer le training

## 🚀 Workflow

1. Agent écrit `trigger.json` dans Drive
2. Notebook détecte et lance training
3. Notebook écrit `status.json` avec progression
4. Checkpoints sauvegardés dans Drive

In [ ]:
# Cell 1 - Montage Drive + vérification GPU
from google.colab import drive
import torch

# Monter Drive
drive.mount('/content/drive')

# Vérifier GPU
print(f"🖥️  GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 - Configuration
import os

# Chemins Drive
DRIVE_BASE = '/content/drive/MyDrive/f5_tts_orchestration'
TRIGGER_PATH = f'{DRIVE_BASE}/trigger.json'
STATUS_PATH = f'{DRIVE_BASE}/status.json'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
LOG_DIR = f'{DRIVE_BASE}/logs'

# Créer répertoires
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Repo F5-TTS
REPO_URL = 'https://github.com/SWivid/F5-TTS.git'

print(f"✅ Configuration:")
print(f"   Trigger: {TRIGGER_PATH}")
print(f"   Status: {STATUS_PATH}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
# Cell 3 - Installation dépendances
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1 sox

# Cloner F5-TTS si pas déjà fait
if not os.path.exists('/content/F5-TTS'):
    print("📥 Clonage F5-TTS...")
    !git clone {REPO_URL} /content/F5-TTS
else:
    print("✅ F5-TTS déjà cloné")

# Installer requirements
%cd /content/F5-TTS
!pip install -q -r requirements.txt

# Dépendances additionnelles
!pip install -q datasets librosa soundfile tqdm tensorboard

print("✅ Installation terminée")

In [ ]:
# Cell 4 - Fonctions utilitaires
import json
import subprocess
import time
from datetime import datetime

def write_status(status_dict):
    """Écrit le statut dans Drive."""
    status_dict['timestamp'] = datetime.now().isoformat()
    with open(STATUS_PATH, 'w', encoding='utf8') as f:
        json.dump(status_dict, f, ensure_ascii=False, indent=2)

def run_training(config):
    """
    Lance le training F5-TTS.
    
    Args:
        config: Dict avec dataset, languages, epochs, etc.
    """
    print("🎓 Démarrage du training...")
    print(f"   Config: {json.dumps(config, indent=2)}")
    
    # Extraire paramètres
    dataset = config.get('dataset', '/content/dataset')
    languages = config.get('languages', ['fr', 'ar'])
    epochs = config.get('epochs', 200)
    batch_size = config.get('batch_size', 1)
    
    # Télécharger dataset depuis GCS si spécifié
    dataset_gs = config.get('dataset_gs')
    if dataset_gs and dataset_gs.startswith('gs://'):
        print(f"📥 Téléchargement dataset: {dataset_gs}")
        !gsutil -m rsync -r {dataset_gs} {dataset}
    
    # Construire commande
    langs_str = ' '.join(languages)
    cmd = [
        'python', 'train.py',
        '--dataset_path', dataset,
        '--multilingual', 'true',
        '--languages', *languages,
        '--epochs', str(epochs),
        '--batch_size', str(batch_size),
        '--device', 'cuda',
        '--checkpoint_dir', CHECKPOINT_DIR,
        '--log_dir', LOG_DIR
    ]
    
    print(f"   Commande: {' '.join(cmd)}")
    
    # Lancer training
    write_status({'status': 'running', 'config': config, 'epoch': 0})
    
    log_file = f"{LOG_DIR}/training_{int(time.time())}.log"
    
    with open(log_file, 'w', encoding='utf8') as logf:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        
        for line in process.stdout:
            print(line, end='')
            logf.write(line)
            
            # Parser epoch/loss si possible
            if 'Epoch' in line:
                try:
                    parts = line.split()
                    epoch = int(parts[parts.index('Epoch')+1].split('/')[0])
                    write_status({
                        'status': 'running',
                        'epoch': epoch,
                        'total_epochs': epochs
                    })
                except:
                    pass
        
        process.wait()
        return_code = process.returncode
    
    # Statut final
    if return_code == 0:
        write_status({'status': 'completed', 'log_file': log_file})
        print("✅ Training terminé avec succès!")
    else:
        write_status({
            'status': 'failed',
            'error': f'Exit code {return_code}',
            'log_file': log_file
        })
        print(f"❌ Training échoué (code: {return_code})")
    
    return return_code

print("✅ Fonctions chargées")

In [ ]:
# Cell 5 - Boucle de polling (LANCER CETTE CELL ET LAISSER TOURNER)
import time
import json
import os

print("👁️  Polling démarré...")
print(f"   Surveillance: {TRIGGER_PATH}")
print(f"   Intervalle: 10s")
print("")
print("⚠️  Laissez cette cell tourner. L'agent écrira trigger.json pour démarrer.")
print("")

poll_count = 0

while True:
    poll_count += 1
    
    # Vérifier trigger
    if os.path.exists(TRIGGER_PATH):
        print(f"\n🚀 Trigger détecté! (poll #{poll_count})")
        
        try:
            # Lire config
            with open(TRIGGER_PATH, 'r', encoding='utf8') as f:
                config = json.load(f)
            
            print(f"   Job ID: {config.get('job_id', 'unknown')}")
            
            # Lancer training
            return_code = run_training(config.get('config', config))
            
            # Supprimer trigger
            try:
                os.remove(TRIGGER_PATH)
                print("   Trigger supprimé")
            except:
                pass
            
            print("\n👁️  Retour en mode polling...\n")
            
        except Exception as e:
            print(f"❌ Erreur: {e}")
            write_status({'status': 'error', 'error': str(e)})
    
    else:
        # Afficher heartbeat toutes les 10 polls
        if poll_count % 10 == 0:
            print(f"💓 Heartbeat #{poll_count} - En attente de trigger...")
    
    # Attendre
    time.sleep(10)

## 📊 Monitoring Manuel

Si vous voulez vérifier le statut manuellement :

In [ ]:
# Cell 6 - Vérifier statut
import json
import os

if os.path.exists(STATUS_PATH):
    with open(STATUS_PATH, 'r') as f:
        status = json.load(f)
    print("📊 Statut actuel:")
    print(json.dumps(status, indent=2, ensure_ascii=False))
else:
    print("⚠️  Aucun statut disponible")

In [ ]:
# Cell 7 - Lister checkpoints
import os

if os.path.exists(CHECKPOINT_DIR):
    checkpoints = os.listdir(CHECKPOINT_DIR)
    print(f"📦 Checkpoints ({len(checkpoints)}):")
    for ckpt in sorted(checkpoints):
        size = os.path.getsize(f"{CHECKPOINT_DIR}/{ckpt}") / 1e6
        print(f"   - {ckpt} ({size:.1f} MB)")
else:
    print("⚠️  Aucun checkpoint")

## 🎯 Test Manuel (Sans Agent)

Pour tester sans agent, créez manuellement un trigger :

In [ ]:
# Cell 8 - Créer trigger de test
import json
import time

test_trigger = {
    "job_id": f"test_{int(time.time())}",
    "status": "pending",
    "config": {
        "languages": ["fr"],
        "dataset": "/content/dataset",  # Assurez-vous qu'il existe
        "epochs": 5,  # Test rapide
        "batch_size": 1
    }
}

with open(TRIGGER_PATH, 'w', encoding='utf8') as f:
    json.dump(test_trigger, f, indent=2, ensure_ascii=False)

print("✅ Trigger de test créé!")
print("   La cell de polling devrait le détecter dans ~10s")